In [21]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [22]:
train_data= pd.read_csv("samsum-train.csv")
val_data= pd.read_csv("samsum-validation.csv")

In [23]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [24]:
train_data.sample(12)

,id,dialogue,summary
4742,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
8870,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
6554,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
12900,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
2596,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."
6422,13716070,Paul: Lucky you!\r\nJohn: ?\r\nPete: Our class...,"John, Pete and Paul's classes have been cancel..."
2452,13727976,Jasper: i miss you so much already :(\r\nKaren...,Karen will be back on Sunday. Karen and Jasper...
476,13681231,Ken: how long do you need?\r\nJude: i think ab...,Ken will wait inside as Jude needs 10 more min...
14716,13862652,"Victoria: Hey, I am in the toilet...And..\nSky...",Victoria is in a restaurant toilet and texts S...
10143,13728508,Sandra: Do u need any help with the party tomo...,Ronda does not need any help with the party to...


In [25]:
train_data.shape

(14732, 3)

In [26]:
val_data.shape

(818, 3)

In [27]:
#Random Sampling

train_data= train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data= val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [28]:
train_data.shape

(4000, 3)

In [29]:
val_data.shape

(500, 3)

In [47]:
# Data Pre_Processing

import re

def clean_data(text):
    text=re.sub(r"\r\n", " ", text) #Lines
    text=re.sub(r"\s+", " ", text) #space
    text=re.sub(r"<.*?>", " ", text) #HTML Tags
    text=text.strip().lower()
    return text

In [48]:
train_data["dialogue"]= train_data["dialogue"].apply(clean_data)
train_data["summary"]= train_data["summary"].apply(clean_data)

val_data["dialogue"]= val_data["dialogue"].apply(clean_data)
val_data["summary"]= val_data["summary"].apply(clean_data)

In [49]:
train_data["dialogue"][0]

"violet: hi! i came acro thi au tin' article and i thought that you might find it intere ting violet: claire: hi! :) thank , but i've already read it. :) claire: but thank for thinking about me :)"

In [50]:
### Tokenize

tokenizer=T5Tokenizer.from_pretrained("t5-small")

#Raw_data=> tokenized input for fine-tuning

def tokenize(data):
    inputs=tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets=tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)
    inputs["labels"]=targets["input_ids"] #token_ids=> add to input as labels
    return inputs


train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [51]:
### Working with our model

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

#Raw_data=> tokenized input for fine-tuning

def tokenize(data):
    inputs=tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets=tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)
    inputs["labels"]=targets["input_ids"] #token_ids=> add to input as labels
    return inputs


train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [52]:
import torch

if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")
print("device: ",device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [53]:
# Training Arguments

training_args= TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500

)

In [54]:
trainer= Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.436326,0.516488
2,0.528022,0.469247
3,0.488924,0.454098


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,3.436326,0.516488
2,0.528022,0.469247
3,0.488924,0.454098
4,0.471714,0.447558
5,0.460139,0.443953
6,0.455427,0.442476


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9734252370198567, metrics={'train_runtime': 1272.272, 'train_samples_per_second': 18.864, 'train_steps_per_second': 2.358, 'total_flos': 3248203235328000.0, 'train_loss': 0.9734252370198567, 'epoch': 6.0})

In [56]:
model.save_pretrained("./save_summary_model")
tokenizer.save_pretrained("./save_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./save_summary_model/tokenizer_config.json',
 './save_summary_model/tokenizer.json')

In [57]:
model=T5ForConditionalGeneration.from_pretrained("./save_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./save_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [58]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) #Clean
    #Tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    #Generate the Summary => token_ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # Token_id convert to summary => Decode
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)
    return summary


In [60]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact. ensuring fairness and transparency is becoming a key area of research.
